In [1]:
# !pip install sacremoses
# !pip install peft

In [1]:
import sys
import os
import pandas as pd
from datasets import Dataset
import torch
from transformers import BioGptTokenizer, BioGptForCausalLM
from transformers import TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType

sys.path.append(os.path.abspath(".."))
from utils.preprocess import load_and_preprocess, upsample_no_and_maybe
from utils.llm_utils import LLMUtils
from utils.evaluation import print_metrics

In [2]:
llmUtils = LLMUtils(("microsoft/BioGPT"))

In [4]:
# Initializing the model and tokenizer
tokenizer = BioGptTokenizer.from_pretrained("microsoft/BioGPT")
model = BioGptForCausalLM.from_pretrained("microsoft/BioGPT")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

In [5]:
# Setting the padding end-of-sequence as padding token if padding token is not there
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

In [6]:
has_gpu = torch.cuda.is_available()

if has_gpu:
    torch.cuda.empty_cache()

device = torch.device("cuda" if has_gpu else "cpu") # My lap has GPU, but making it as dynamic, so you can run with cpu as well.
model.to(device)
model.eval()

BioGptForCausalLM(
  (biogpt): BioGptModel(
    (embed_tokens): BioGptScaledWordEmbedding(42384, 1024, padding_idx=1)
    (embed_positions): BioGptLearnedPositionalEmbedding(1026, 1024)
    (layers): ModuleList(
      (0-23): 24 x BioGptDecoderLayer(
        (self_attn): BioGptAttention(
          (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
        )
        (activation_fn): GELUActivation()
        (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (fc1): Linear(in_features=1024, out_features=4096, bias=True)
        (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        (final_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      )
    )
    (layer_norm): LayerNorm((1024

In [7]:
# Load data
train_df = load_and_preprocess('../Preprocessing+baseline/data/ori_pqal.json', False)
test_df = load_and_preprocess('../Preprocessing+baseline/data/test_set.json', False)
train_df.head()

,pmid,text,question,context,label
0,21645374,Do mitochondria play a role in remodelling lac...,Do mitochondria play a role in remodelling lac...,Programmed cell death (PCD) is the regulated d...,yes
1,16418930,Landolt C and snellen e acuity: differences in...,Landolt C and snellen e acuity: differences in...,Assessment of visual acuity depends on the opt...,no
2,9488747,"Syncope during bathing in infants, a pediatric...","Syncope during bathing in infants, a pediatric...",Apparent life-threatening events in infants ar...,yes
3,17208539,Are the long-term results of the transanal pul...,Are the long-term results of the transanal pul...,The transanal endorectal pull-through (TERPT) ...,no
4,10808977,Can tailored interventions increase mammograph...,Can tailored interventions increase mammograph...,Telephone counseling and tailored print commun...,yes


In [8]:
upsampled_df = upsample_no_and_maybe(test_df)

In [9]:
upsampled_df['label'].value_counts()

label
maybe    276
yes      276
no       276
Name: count, dtype: int64

In [10]:
# Tokenize the the training data
upsampled_df['tokens'] = upsampled_df.apply(llmUtils.tokenize_data, axis=1)
upsampled_df['tokens']

477    {'input_ids': [4790, 20925, 20, 10964, 10057, ...
457    {'input_ids': [4790, 20925, 20, 8928, 5305, 60...
462    {'input_ids': [4790, 20925, 20, 2882, 223, 14,...
174    {'input_ids': [4790, 20925, 20, 2882, 4970, 55...
484    {'input_ids': [4790, 20925, 20, 24235, 463, 9,...
                             ...                        
259    {'input_ids': [4790, 20925, 20, 10552, 23161, ...
32     {'input_ids': [4790, 20925, 20, 9260, 1632, 5,...
298    {'input_ids': [4790, 20925, 20, 8928, 14967, 1...
408    {'input_ids': [4790, 20925, 20, 2882, 1422, 24...
257    {'input_ids': [4790, 20925, 20, 10552, 60, 798...
Name: tokens, Length: 828, dtype: object

In [11]:
# Create dataframe from the tokens
tokenized_df = pd.DataFrame(upsampled_df["tokens"].tolist())
tokenized_df

,input_ids,attention_mask,labels
0,"[4790, 20925, 20, 10964, 10057, 1309, 17826, 5...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, -100, -100, -100, -100, -100, -100, -10..."
1,"[4790, 20925, 20, 8928, 5305, 600, 5, 23032, 3...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, -100, -100, -100, -100, -100, -100, -10..."
2,"[4790, 20925, 20, 2882, 223, 14, 151, 16, 7380...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, -100, -100, -100, -100, -100, -100, -10..."
3,"[4790, 20925, 20, 2882, 4970, 5595, 24272, 126...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, -100, -100, -100, -100, -100, -100, -10..."
4,"[4790, 20925, 20, 24235, 463, 9, 798, 28, 33, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, -100, -100, -100, -100, -100, -100, -10..."
...,...,...,...
823,"[4790, 20925, 20, 10552, 23161, 754, 5, 33651,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, -100, -100, -100, -100, -100, -100, -10..."
824,"[4790, 20925, 20, 9260, 1632, 5, 5239, 204, 20...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, -100, -100, -100, -100, -100, -100, -10..."
825,"[4790, 20925, 20, 8928, 14967, 13710, 8, 8545,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, -100, -100, -100, -100, -100, -100, -10..."
826,"[4790, 20925, 20, 2882, 1422, 245, 1378, 14, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, -100, -100, -100, -100, -100, -100, -10..."


In [12]:
# Create dataset from the tokenized dataframe
train_dataset = Dataset.from_pandas(tokenized_df[["input_ids", "attention_mask", "labels"]])
train_dataset

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 828
})

In [14]:
# Configuring LoRA to do parameter-efficient fine-tuning
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

c:\Users\ruthr\anaconda3\Lib\site-packages\peft\mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
c:\Users\ruthr\anaconda3\Lib\site-packages\peft\tuners\tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 1,572,864 || all params: 348,336,128 || trainable%: 0.4515


In [15]:
training_args = TrainingArguments(
    output_dir="./biogpt-lora-pubmedqa",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    learning_rate=2e-4,
    weight_decay=0.01,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    fp16=has_gpu,
    report_to="none",
)

In [16]:
from torch.nn.utils.rnn import pad_sequence

def custom_collate(batch):
    input_ids = [torch.tensor(x["input_ids"], dtype=torch.long) for x in batch]
    attention_mask = [torch.tensor(x["attention_mask"], dtype=torch.long) for x in batch]
    labels = [torch.tensor(x["labels"], dtype=torch.long) for x in batch]

    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_mask = pad_sequence(attention_mask, batch_first=True, padding_value=0)
    labels = pad_sequence(labels, batch_first=True, padding_value=-100)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

In [17]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=custom_collate,
)

In [18]:
trainer.train()

c:\Users\ruthr\anaconda3\Lib\site-packages\transformers\models\biogpt\modeling_biogpt.py:377: FutureWarning: `input_embeds` is deprecated and will be removed in version 5.6.0 for `create_causal_mask`. Use `inputs_embeds` instead.
  causal_mask = create_causal_mask(


Step,Training Loss
50,1.919166
100,1.477509
150,1.143649
200,1.491149
250,1.077157
300,1.014277
350,1.138346
400,1.098403
450,1.078356
500,0.849702


c:\Users\ruthr\anaconda3\Lib\site-packages\transformers\models\biogpt\modeling_biogpt.py:377: FutureWarning: `input_embeds` is deprecated and will be removed in version 5.6.0 for `create_causal_mask`. Use `inputs_embeds` instead.
  causal_mask = create_causal_mask(
c:\Users\ruthr\anaconda3\Lib\site-packages\transformers\models\biogpt\modeling_biogpt.py:377: FutureWarning: `input_embeds` is deprecated and will be removed in version 5.6.0 for `create_causal_mask`. Use `inputs_embeds` instead.
  causal_mask = create_causal_mask(


TrainOutput(global_step=1242, training_loss=0.8870693320429459, metrics={'train_runtime': 225.554, 'train_samples_per_second': 11.013, 'train_steps_per_second': 5.506, 'total_flos': 1576032651780096.0, 'train_loss': 0.8870693320429459, 'epoch': 3.0})

In [19]:
def generate_answer(question, context, max_new_tokens=20):
    prompt = llmUtils.build_prompt(question, context)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            # num_beams=1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    full_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    # extracting only the newly generated text from the full text
    generated_text = full_text[len(prompt):].strip()
    return generated_text

In [20]:
predicted_values = []
actual_values = []

In [21]:
for index, row in test_df.iterrows():
    # normalizing the labels of the test data
    actual_values.append(llmUtils.normalize_label(row["label"]))

    # generating and normalizing the labels of the generated data
    generated_answer = generate_answer(row["question"], row["context"])
    predicted_values.append(llmUtils.normalize_label(generated_answer))

c:\Users\ruthr\anaconda3\Lib\site-packages\transformers\models\biogpt\modeling_biogpt.py:377: FutureWarning: `input_embeds` is deprecated and will be removed in version 5.6.0 for `create_causal_mask`. Use `inputs_embeds` instead.
  causal_mask = create_causal_mask(


In [23]:
test_df.head()

,pmid,text,question,context,label
0,12377809,Is anorectal endosonography valuable in dysche...,Is anorectal endosonography valuable in dysche...,Dyschesia can be provoked by inappropriate def...,yes
1,26163474,Is there a connection between sublingual varic...,Is there a connection between sublingual varic...,Sublingual varices have earlier been related t...,yes
2,19100463,Is the affinity column-mediated immunoassay me...,Is the affinity column-mediated immunoassay me...,Tacrolimus is a potent immunosuppressive drug ...,yes
3,18537964,Does a physician's specialty influence the rec...,Does a physician's specialty influence the rec...,To determine the impact of a physician's speci...,yes
4,12913878,Locoregional opening of the rodent blood-brain...,Locoregional opening of the rodent blood-brain...,Nd:YAG laser-induced thermo therapy (LITT) of ...,yes


In [27]:
results = test_df[["pmid"]]
results.rename(columns={"pmid": "pubmed_id"})
results["y_true"] = pd.Series(actual_values)
results["y_pred"] = pd.Series(predicted_values)
results.head()

C:\Users\ruthr\AppData\Local\Temp\ipykernel_62432\3396944186.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results["y_true"] = pd.Series(actual_values)
C:\Users\ruthr\AppData\Local\Temp\ipykernel_62432\3396944186.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results["y_pred"] = pd.Series(predicted_values)


,pmid,y_true,y_pred
0,12377809,yes,yes
1,26163474,yes,yes
2,19100463,yes,yes
3,18537964,yes,yes
4,12913878,yes,yes


In [29]:
import numpy as np

results["correct"] = np.where(results["y_true"] == results["y_pred"], 1, 0)

C:\Users\ruthr\AppData\Local\Temp\ipykernel_62432\1463632195.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results["correct"] = np.where(results["y_true"] == results["y_pred"], 1, 0)


In [32]:
results.tail()

,pmid,y_true,y_pred,correct
495,17691856,maybe,yes,0
496,16735905,maybe,yes,0
497,19694846,maybe,yes,0
498,25007420,maybe,yes,0
499,26134053,maybe,yes,0


In [ ]:
results.to_csv("results.csv", index=False)

In [22]:
print_metrics(actual_values, predicted_values)

Accuracy 0.674
Macro F1 0.5580100250373711
Classification_report

              precision    recall  f1-score   support

         yes       0.65      0.94      0.77       276
          no       0.74      0.38      0.50       169
       maybe       1.00      0.25      0.41        55

    accuracy                           0.67       500
   macro avg       0.80      0.52      0.56       500
weighted avg       0.72      0.67      0.64       500

